# 01 数据清洗与经营总览

> 本 notebook 展示从 Olist 原始 9 表到订单粒度事实表的完整清洗流程，以及经营层面的 EDA。
>
> **核心原则：先按订单粒度聚合事实表，再算 GMV——杜绝一对多 JOIN 后直接求和导致的重复计算。**
>
> 数据模式：默认读取 `dashboard/data/` 下的真实分析输出（OLIST_ACTUAL）；如需从原始 CSV 重跑，执行 `src/run_olist.py`。

## 1. 环境与路径配置

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'dashboard' / 'data'
RAW_DIR = ROOT / 'data' / 'raw'

# 确认数据模式
with open(DATA_DIR / 'summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)
print(f"数据模式: {summary['data_mode']}")
print(f"数据截止: {summary['data_max_purchase_date']}")
print(f"有效订单: {summary['valid_orders']:,} | 用户: {summary['valid_users']:,} | GMV: {summary['gmv']:,.2f}")

## 2. 原始数据概览

Olist 数据集包含 9 张表，核心关联关系：
- `orders` —— 订单主表（order_id, customer_id, 状态、时间戳）
- `customers` —— 用户表（customer_id → customer_unique_id，一个用户可有多订单）
- `order_items` —— 订单商品行（**一对多**，一个订单可含多商品）
- `order_payments` —— 支付记录（**一对多**，可分次支付）
- `order_reviews` —— 评价（一对多）
- `products` —— 商品信息（含品类）

In [ ]:
# 如果原始 CSV 存在，展示各表规模
raw_files = {
    'orders': 'olist_orders_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
}

for name, fname in raw_files.items():
    fpath = RAW_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f"{name:12s} {len(df):>8,} 行  {df.shape[1]:>2d} 列")
    else:
        print(f"{name:12s} (原始文件未放置，使用已生成的事实表)")

## 3. 数据质量检查

在进入分析前，必须确认：
1. 各表主键无重复、无空值
2. 订单与用户关联完整
3. 已交付订单金额为正
4. 时间逻辑合理（交付时间不早于下单时间）

In [ ]:
# 读取已生成的质量检查表
quality = pd.read_csv(DATA_DIR / 'data_quality.csv')
print("=== 表级检查 ===")
print(quality[quality['check_type'] == 'table'][['check_name', 'rows', 'duplicate_rows', 'null_primary_key']].to_string(index=False))
print()
print("=== 业务规则检查 ===")
biz = quality[quality['check_type'] == 'business']
print(biz[['check_name', 'failed_rows']].to_string(index=False))
print()
print("结论：所有业务检查失败行数均为 0，数据可进入分析。" if biz['failed_rows'].sum() == 0 else "警告：存在业务检查失败，需先处理。")

## 4. 构建订单粒度事实表（核心步骤）

**为什么不能直接 JOIN 后求和？**

`order_items` 和 `order_payments` 都是一对多表。如果直接：
```sql
SELECT o.order_id, SUM(i.price + i.freight) AS gmv
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
JOIN order_payments p ON o.order_id = p.order_id
```
商品行会被支付行笛卡尔放大，GMV 被重复计算。

**正确做法：先分别按 order_id 聚合，再一对一 JOIN。**

In [ ]:
# 演示事实表构建逻辑（与 src/analysis.py 中 load_and_clean 一致）
if (RAW_DIR / 'olist_orders_dataset.csv').exists():
    orders = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv',
                         parse_dates=['order_purchase_timestamp',
                                      'order_delivered_customer_date',
                                      'order_estimated_delivery_date'])
    customers = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
    items = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
    payments = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')
    reviews = pd.read_csv(RAW_DIR / 'olist_order_reviews_dataset.csv')
    products = pd.read_csv(RAW_DIR / 'olist_products_dataset.csv',
                           usecols=['product_id', 'product_category_name'])

    # 商品表：先按 order_id 聚合
    items = items.merge(products, on='product_id', how='left')
    item_agg = items.groupby('order_id', as_index=False).agg(
        product_amount=('price', 'sum'),
        freight_amount=('freight_value', 'sum'),
        item_count=('order_item_id', 'count'),
        category=('product_category_name', lambda x: x.mode().iat[0] if not x.mode().empty else 'unknown')
    )

    # 支付表：取最大支付金额的记录作为支付方式代表
    pay_agg = (payments.sort_values('payment_value', ascending=False)
               .drop_duplicates('order_id')[['order_id', 'payment_type', 'payment_installments']])

    # 评价：按订单取均分
    review_agg = reviews.groupby('order_id', as_index=False).agg(
        review_score=('review_score', 'mean')
    )

    # 一对一 JOIN 构建事实表
    fact = (orders
            .merge(customers, on='customer_id', how='left', validate='one_to_one')
            .merge(item_agg, on='order_id', how='left', validate='one_to_one')
            .merge(pay_agg, on='order_id', how='left', validate='one_to_one')
            .merge(review_agg, on='order_id', how='left', validate='one_to_one'))

    fact['gmv'] = fact['product_amount'].fillna(0) + fact['freight_amount'].fillna(0)
    fact['purchase_month'] = fact['order_purchase_timestamp'].dt.to_period('M').astype(str)
    fact['delivery_days'] = (fact['order_delivered_customer_date'] - fact['order_purchase_timestamp']).dt.total_seconds() / 86400
    fact['late_delivery'] = np.where(
        fact['order_delivered_customer_date'].notna(),
        fact['order_delivered_customer_date'] > fact['order_estimated_delivery_date'],
        np.nan
    )
    fact['is_valid_analysis'] = (
        fact['order_status'].eq('delivered') &
        fact['gmv'].gt(0) &
        fact['order_purchase_timestamp'].notna()
    )

    print(f"事实表行数: {len(fact):,}")
    print(f"有效分析订单: {fact['is_valid_analysis'].sum():,}")
    print(f"事实表 order_id 唯一性: {fact['order_id'].is_unique}")
else:
    # 原始文件不存在时，直接读取已生成的事实表
    fact = pd.read_csv(DATA_DIR / 'order_fact.csv',
                       parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date'])
    print(f"从已生成输出读取事实表: {len(fact):,} 行")

## 5. 经营总览 EDA

In [ ]:
valid = fact[fact['is_valid_analysis']].copy()
print(f"=== 经营核心指标 ===")
print(f"有效订单数:   {len(valid):,}")
print(f"独立用户数:   {valid['customer_unique_id'].nunique():,}")
print(f"GMV:          R$ {valid['gmv'].sum():,.2f}")
print(f"客单价(AOV):  R$ {valid['gmv'].sum() / len(valid):,.2f}")
print(f"观察窗复购率: {(valid.groupby('customer_unique_id')['order_id'].nunique() >= 2).mean():.1%}")

In [ ]:
# 月度趋势（排除不完整月）
monthly = pd.read_csv(DATA_DIR / 'monthly_kpis.csv')
complete = monthly[monthly['is_complete_month'] == True].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(complete['purchase_month'], complete['gmv'], marker='o', linewidth=2)
axes[0].set_title('月度 GMV 趋势')
axes[0].set_ylabel('GMV (R$)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(complete['purchase_month'], complete['users'], marker='s', linewidth=2, color='orange')
axes[1].set_title('月度活跃用户数')
axes[1].set_ylabel('用户数')
axes[1].tick_params(axis='x', rotation=45)

axes[2].plot(complete['purchase_month'], complete['aov'], marker='^', linewidth=2, color='green')
axes[2].set_title('月度客单价 (AOV)')
axes[2].set_ylabel('AOV (R$)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# 展示最近完整月的三因子
latest = complete.iloc[-1]
print(f"最近完整月 ({latest['purchase_month']}):")
print(f"  GMV={latest['gmv']:,.2f}  用户={latest['users']:,}  频次={latest['orders_per_user']:.4f}  AOV={latest['aov']:.2f}")
print(f"  恒等式自检 identity_gap = {latest['identity_gap']:.2e} (应≈0)")

In [ ]:
# 品类分布 TOP10
category = pd.read_csv(DATA_DIR / 'category_kpis.csv')
top10_cat = category.head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top10_cat['category'][::-1], top10_cat['gmv'][::-1], color='steelblue')
ax.set_title('GMV TOP10 品类')
ax.set_xlabel('GMV (R$)')
for bar, val in zip(bars, top10_cat['gmv'][::-1]):
    ax.text(val, bar.get_y() + bar.get_height()/2, f' {val:,.0f}', va='center')
plt.tight_layout()
plt.show()

print(top10_cat[['category', 'gmv', 'orders', 'aov', 'avg_review']].to_string(index=False))

In [ ]:
# 地区分布 TOP10
state = pd.read_csv(DATA_DIR / 'state_kpis.csv')
top10_state = state.head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(top10_state['customer_state'], top10_state['gmv'], color='coral')
ax.set_title('GMV TOP10 州')
ax.set_ylabel('GMV (R$)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 支付方式分布
payment = pd.read_csv(DATA_DIR / 'payment_kpis.csv')
payment = payment.sort_values('gmv', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(payment['orders'], labels=payment['payment_type'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('支付方式按订单数占比')
axes[1].pie(payment['gmv'], labels=payment['payment_type'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('支付方式按 GMV 占比')
plt.tight_layout()
plt.show()

## 6. 履约体验初步诊断

逾期交付与用户评分的关系是后续运营策略的重要依据。
注意：此处仅为**描述性相关性**，不能直接推断因果。

In [ ]:
delivery = pd.read_csv(DATA_DIR / 'delivery_impact.csv')
print(delivery.to_string(index=False))
print()

late = delivery[delivery['late_delivery'] == True].iloc[0]
ontime = delivery[delivery['late_delivery'] == False].iloc[0]
print(f"逾期订单: {late['orders']:,} 单，平均评分 {late['avg_review']:.2f}")
print(f"准时订单: {ontime['orders']:,} 单，平均评分 {ontime['avg_review']:.2f}")
print(f"评分差: {ontime['avg_review'] - late['avg_review']:.2f} 分（仅相关性，非因果）")

## 7. 小结

- 数据质量全部通过，事实表按订单粒度构建，GMV 无重复计算风险。
- 经营层面：增长存在但动能有限，复购率仅 3.0% 是核心约束。
- 履约逾期订单评分显著低于准时订单，是 P0 优先级的体验杠杆。
- 下一步进入用户分层与留存分析（见 `02_user_rfm_cohort.ipynb`）。